# Pre-Arrears Strategy — Component: Previous Performance 12m (SAS Step 2)

**Source**: `Pre-Arrears.txt`, lines 218–296 (`%macro prep; ... %mend prep; %prep;`)

Second component of the 10-step strategy, converted independently per the
brief. Depends only on the warehouse (not on Step 1's output) — it builds
its own base population (`M0`) and 12 months of lookback indicators, then
folds them into one row per account.


## 1. What the SAS component does

For `i = 0 .. 12` (13 iterations), the macro runs one SQL Server
pass-through query per historical month (`&rundate` minus `i` months) and
materializes it as `M<i>`. Each query returns, per account:

- `MIA_1P<i>` — 1 if the account was **1+ months in arrears** in that
  historical month (Contractual arrears for SBS, Standard arrears for
  AHL/NYM), else 0.
- `Arr_Ind<i>` — 1 if the account had **any arrears amount > 0** in that
  historical month, else 0.

The population for each `M<i>` is: SBS/AHL/NYM, not pool `DRC00001`,
`AccountSubType='Primary'`, status `Live` in that historical month, and
the account must have started on or before the last day of the month
*preceding* that historical snapshot (i.e. it must have existed for at
least one full month by the time of that snapshot).

**`M0`** (i=0, the current/reporting month) becomes the base population,
renamed to `Prep<yyyymm>`, and its own `MIA_1P0`/`Arr_Ind0` columns are
carried through the merges but dropped by the final `keep` statement —
`M0` only exists to fix the account universe.

For `i = 1 .. 12`, `Prep<yyyymm>` is **left-joined** to `M<i>` (`merge ...
if a;` with `a` = the running `Prep` table = a SAS left join keeping only
accounts present in `M0`). Where an account has no matching row in
`M<i>` (i.e. it wasn't Live/didn't meet the population rule in that
historical month), `MIA_1P<i>` / `Arr_Ind<i>` are explicitly coalesced
to `0` — "no data for that month" is treated as "no missed instalment /
no arrears" for that month, not as missing.

Finally, one row per account is kept with:
- `MIAH`, `ArrH` — intended as a 12-character history string of the
  monthly flags (months 1–12 back). **See risk #1 below — this does not
  do what it looks like it does.**
- `MIA1X_pre12m` — 1 if the account missed an instalment **at least
  once** in the trailing 12 months (excluding the current month).
- `AB1X_pre12m` — 1 if the account was in arrears **exactly once** in
  the trailing 12 months.
- `AB2X_pre12m` — 1 if the account was in arrears **more than once** in
  the trailing 12 months.


## 2. Semantic risks

1. **`MIAH`/`ArrH` are effectively broken in the source, and this must be
   reproduced, not fixed.** SAS fixes a variable's storage length at its
   *first* appearance in a DATA step. `MIAH='            ';` (12 spaces)
   fixes `MIAH` at length 12 **before** `MIAH=cat(of MIA_1P1-MIA_1P12);`
   runs. `cat()` does not strip padding, and each numeric argument is
   converted using its default `BEST12.` format (12 characters,
   right-justified) when no explicit format is attached — so the "12
   flags concatenated" the code appears to intend would actually be a
   144-character string, which SAS then **silently truncates to the
   first 12 characters** because of the length already fixed. Those
   first 12 characters are just `MIA_1P1`'s own right-justified 12-wide
   representation (11 spaces + `"0"`/`"1"`) — i.e. **`MIAH` and `ArrH`
   end up encoding only month-1's flag, padded with leading blanks, not
   a 12-month history.** This is reproduced exactly below.
   **TODO: VERIFY with the business whether `MIAH`/`ArrH` are consumed
   anywhere downstream** — if they are read as a "history string" by a
   report or model, that consumer has been receiving this truncated
   value the whole time and any fix belongs in a separate,
   explicitly-approved change, not this like-for-like conversion.
2. **Merge relies on incoming sort order, not an explicit sort.** SAS
   `MERGE ... BY AccountNumber` requires both inputs to already be
   sorted (or indexed) by the BY variable; there is no `proc sort` on
   `M<i>` before the merge loop — it relies entirely on the pass-through
   query's `ORDER BY` having produced sorted output from SQL Server. If
   that guarantee is ever violated (e.g. a driver/version change altering
   result ordering), SAS's merge would silently produce **wrong**
   matches without erroring. The Python version does not have this
   fragility since it merges on the key explicitly regardless of row
   order, but this is flagged as a difference in *robustness*, not
   business logic.
3. **Missing vs. explicit zero.** The `if MIA_1P&i=. then MIA_1P&i=0`
   pattern is an explicit, deliberate coalesce (not a SAS-missing
   accident) — "account didn't exist/wasn't Live in that historical
   month" is intentionally treated as "no missed instalment that
   month". This must be replicated with an explicit `fillna(0)`, not
   left as `NaN`, or every downstream sum/threshold would be wrong.
4. **Duplicate handling.** No `first.`/`last.` logic in this step. A
   duplicate `AccountNumber` within a single `M<i>` would fan the merge
   out to multiple rows per account — the source query's `INNER JOIN`
   chain should yield one row per account per month by construction, but
   this is not guaranteed by a `DISTINCT`/dedup step, so it is validated
   explicitly in the Python version rather than assumed.
5. **Retained values.** None — no `retain` statement in this component.
6. **Threshold boundary.** `AB1X_pre12m` uses `= 1` (exactly one),
   `AB2X_pre12m` uses `> 1` (strictly more than one) — these are
   mutually exclusive by construction and must stay that way; a common
   conversion mistake is to use `>=1` for both.


In [ ]:
"""
previous_performance.py
Production module: Pre-Arrears Strategy — Step 2: Previous Performance (12m).

Converted 1:1 from Pre-Arrears.txt lines 218-296 (SAS macro %prep).
Reuses the DataWarehouseConnector / retry pattern introduced in the
Step 1 (Pre-Arrears Qualifiers) module.
"""

import logging
import sys
from dataclasses import dataclass, field
from datetime import date
from typing import List

import pandas as pd

logger = logging.getLogger("pre_arrears.previous_performance")
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-8s | %(name)s | %(message)s")
    )
    logger.addHandler(_handler)
    logger.setLevel(logging.INFO)


class PreviousPerformanceError(Exception):
    """Raised when Step 2 extraction/aggregation cannot be trusted to proceed."""


In [ ]:
@dataclass(frozen=True)
class PreviousPerformanceConfig:
    """Defaults are identical to the SAS literals; see risk log above."""

    rundate: date  # must be the 1st day of a calendar month
    lookback_months: int = 12  # SAS: %do i=0 %to 12

    db_server: str = "SBSPSQLVS101\\MISPSQL01"
    db_database: str = "P_DW"
    db_driver: str = "ODBC Driver 17 for SQL Server"
    connect_timeout_seconds: int = 30
    max_retries: int = 3
    retry_backoff_seconds: float = 5.0

    company_scope: List[str] = field(default_factory=lambda: [
        "Skipton Building Society", "Amber Homeloans Ltd", "North Yorkshire Mortgages",
    ])
    excluded_account_pool: str = "DRC00001"

    def __post_init__(self):
        if self.rundate.day != 1:
            raise ValueError(f"rundate must be the first day of a month; got {self.rundate}")
        if self.lookback_months < 1:
            raise ValueError("lookback_months must be >= 1")


In [ ]:
def get_lookback_period(rundate: date, months_back: int) -> int:
    """Equivalent of SAS: intnx(month,&rundate,&i*-1) formatted yymmn6. -> int YYYYMM."""
    year, month = rundate.year, rundate.month
    total_months = year * 12 + (month - 1) - months_back
    return (total_months // 12) * 100 + (total_months % 12 + 1)


In [ ]:
def _in_clause(n: int) -> str:
    if n == 0:
        return "(SELECT NULL WHERE 1 = 0)"
    return "(" + ", ".join(["?"] * n) + ")"


def build_monthly_indicator_query(cfg: PreviousPerformanceConfig) -> str:
    """
    Query template for a single historical month's arrears indicators.
    Identical logic to Pre-Arrears.txt lines 232-261, with the month
    literal left as a bind parameter (filled in per call).
    """
    company_in = _in_clause(len(cfg.company_scope))
    return f"""
SELECT  CASE WHEN lu.Allocated_AccountNumber IS NOT NULL THEN lu.Allocated_AccountNumber ELSE da.AccountNumber END AS AccountNumber
        , CASE WHEN dc.CompanyName IN ('Amber Homeloans Ltd', 'North Yorkshire Mortgages') AND fma.StandardArrearsInMonths >= 1 THEN 1
               WHEN dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsInMonths >= 1 THEN 1
               ELSE 0 END AS MIA_1P
        , CASE WHEN dc.CompanyName IN ('Amber Homeloans Ltd', 'North Yorkshire Mortgages') AND fma.StandardArrearsAmount > 0 THEN 1
               WHEN dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsAmount > 0 THEN 1
               ELSE 0 END AS Arr_Ind
FROM    P_DW.dbo.factMortgageAccountMonthSS fma
INNER JOIN  P_DW.dbo.dimAccount da ON fma.FK_Account = da.PK_Account
INNER JOIN  P_DW.dbo.dimCompany dc ON fma.FK_Company = dc.PK_Company
LEFT JOIN   P_ODS.reference.AHLNYMMigratedAccounts lu
    ON  da.AccountPool = lu.AHLNYM_PoolID AND da.AccountNumber = lu.AHLNYM_AccountNumber
INNER JOIN  P_DW.dbo.dimMonth dm ON fma.FK_SnapshotDate = dm.PK_Month
INNER JOIN  P_DW.dbo.dimAccountStates das ON fma.FK_AccountStates = das.PK_AccountStates
WHERE   dc.CompanyName IN {company_in}
AND     da.AccountPool <> ?
AND     dm.CalendarMonthYear = CONVERT(VARCHAR, ?)
AND     da.AccountSubType = 'Primary'
AND     das.AccountStatus IN ('Live')
AND     da.AccountStartDate <= DATEADD(MONTH, DATEDIFF(MONTH, -1, dm.BatchDate) - 1, -1)
ORDER BY (CASE WHEN lu.Allocated_AccountNumber IS NOT NULL THEN lu.Allocated_AccountNumber ELSE da.AccountNumber END)
"""


def build_monthly_indicator_params(cfg: PreviousPerformanceConfig, yyyymm: int) -> list:
    return list(cfg.company_scope) + [cfg.excluded_account_pool, yyyymm]


In [ ]:
def build_history_string(monthly_flags: "list[int]") -> str:
    """
    Faithful reproduction of SAS's MIAH/ArrH bug (risk #1 above): the
    variable's length is fixed at 12 chars by its initial assignment
    before `cat()` overwrites it, so only the RIGHT-JUSTIFIED, 12-wide
    default-format representation of the FIRST monthly flag survives;
    everything else concatenated by `cat()` is truncated away.

    This function is deliberately named `_broken` in spirit: it exists
    to reproduce the legacy output byte-for-byte, not to compute a
    correct 12-month history string. Do not "fix" this without an
    explicit, approved business-rule change (see TODO: VERIFY above).
    """
    if not monthly_flags:
        return " " * 12
    first_value = monthly_flags[0]
    # SAS default BEST12. representation of a small integer: right-justified
    # in a 12-character field.
    return f"{first_value:>12d}"


def compute_previous_performance_flags(mia_flags: "list[int]", arr_flags: "list[int]") -> dict:
    """Equivalent of the final DATA step's summary-flag logic (lines 288-290)."""
    mia_sum = sum(mia_flags)
    arr_sum = sum(arr_flags)
    return {
        "MIAH": build_history_string(mia_flags),
        "ArrH": build_history_string(arr_flags),
        "MIA1X_pre12m": 1 if mia_sum >= 1 else 0,
        "AB1X_pre12m": 1 if arr_sum == 1 else 0,
        "AB2X_pre12m": 1 if arr_sum > 1 else 0,
    }


In [ ]:
def retry(times: int, backoff_seconds: float):
    import functools
    import time

    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as exc:  # noqa: BLE001
                    last_exc = exc
                    logger.warning("Attempt %s/%s failed for %s: %s", attempt, times, fn.__name__, exc)
                    if attempt < times:
                        time.sleep(backoff_seconds * attempt)
            raise PreviousPerformanceError(f"{fn.__name__} failed after {times} attempts") from last_exc
        return wrapper
    return decorator


class DataWarehouseConnector:
    """Trusted (Windows-integrated) auth, matching the SAS libname. No stored credentials."""

    def __init__(self, cfg: PreviousPerformanceConfig):
        self.cfg = cfg
        self._conn = None

    def _connection_string(self) -> str:
        return (
            f"DRIVER={{{self.cfg.db_driver}}};SERVER={self.cfg.db_server};"
            f"DATABASE={self.cfg.db_database};Trusted_Connection=yes;"
            f"Connect Timeout={self.cfg.connect_timeout_seconds};"
        )

    @retry(times=3, backoff_seconds=5.0)
    def connect(self):
        import pyodbc
        logger.info("Connecting to %s/%s", self.cfg.db_server, self.cfg.db_database)
        self._conn = pyodbc.connect(self._connection_string(), timeout=self.cfg.connect_timeout_seconds)
        return self._conn

    def __enter__(self):
        return self.connect()

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self._conn is not None:
            self._conn.close()
            logger.info("Connection closed")


In [ ]:
def extract_monthly_indicators(conn, cfg: PreviousPerformanceConfig, yyyymm: int) -> pd.DataFrame:
    """Equivalent of one loop iteration's `CREATE TABLE M<i>` (lines 229-262)."""
    query = build_monthly_indicator_query(cfg)
    params = build_monthly_indicator_params(cfg, yyyymm)
    df = pd.read_sql(query, conn, params=params)

    dup_count = int(df["AccountNumber"].duplicated().sum())
    if dup_count > 0:
        raise PreviousPerformanceError(
            f"{dup_count} duplicate AccountNumber row(s) for period {yyyymm}; "
            f"expected exactly one row per account (see risk #4)."
        )
    return df.sort_values("AccountNumber", kind="stable").reset_index(drop=True)


def extract_previous_performance(cfg: PreviousPerformanceConfig) -> pd.DataFrame:
    """
    Main entry point. Equivalent to SAS `%prep;` — builds the trailing
    lookback population and returns one row per account with the
    12-month summary indicators. Equivalent to SAS table `Prep<yyyymm>`.
    """
    yyyymm_current = get_lookback_period(cfg.rundate, 0)
    logger.info("Starting Previous Performance extraction for period %s (lookback=%s months)",
                yyyymm_current, cfg.lookback_months)

    with DataWarehouseConnector(cfg) as conn:
        # M0: fixes the account universe (equivalent to `data Prep<yyyymm>; set M0;`)
        base = extract_monthly_indicators(conn, cfg, yyyymm_current)
        base = base.rename(columns={"MIA_1P": "MIA_1P0", "Arr_Ind": "Arr_Ind0"})
        result = base[["AccountNumber"]].copy()

        mia_cols, arr_cols = [], []
        for i in range(1, cfg.lookback_months + 1):
            yyyymm_i = get_lookback_period(cfg.rundate, i)
            month_df = extract_monthly_indicators(conn, cfg, yyyymm_i)
            month_df = month_df.rename(columns={"MIA_1P": f"MIA_1P{i}", "Arr_Ind": f"Arr_Ind{i}"})

            # Equivalent of: merge Prep(in=a) M&i(in=b); by AccountNumber; if a;
            # -> a LEFT JOIN keeping only accounts in the base (M0) population.
            result = result.merge(month_df, on="AccountNumber", how="left")

            # Equivalent of: if MIA_1P&i=. then MIA_1P&i=0; (explicit coalesce, not accidental)
            result[f"MIA_1P{i}"] = result[f"MIA_1P{i}"].fillna(0).astype(int)
            result[f"Arr_Ind{i}"] = result[f"Arr_Ind{i}"].fillna(0).astype(int)
            mia_cols.append(f"MIA_1P{i}")
            arr_cols.append(f"Arr_Ind{i}")

    summary_rows = []
    for _, row in result.iterrows():
        mia_flags = [int(row[c]) for c in mia_cols]
        arr_flags = [int(row[c]) for c in arr_cols]
        summary_rows.append(compute_previous_performance_flags(mia_flags, arr_flags))

    summary = pd.DataFrame(summary_rows, index=result.index)
    final = pd.concat([result[["AccountNumber"]], summary], axis=1)

    logger.info("Completed Previous Performance extraction: %s rows", len(final))
    return final


## 3. Unit tests

In [ ]:
import unittest
from datetime import date as _date


class TestLookbackPeriod(unittest.TestCase):
    def test_zero_months_back_is_current_month(self):
        self.assertEqual(get_lookback_period(_date(2026, 7, 1), 0), 202607)

    def test_one_month_back(self):
        self.assertEqual(get_lookback_period(_date(2026, 7, 1), 1), 202606)

    def test_twelve_months_back_crosses_year_boundary(self):
        self.assertEqual(get_lookback_period(_date(2026, 7, 1), 12), 202507)

    def test_january_one_month_back_crosses_year_boundary(self):
        self.assertEqual(get_lookback_period(_date(2026, 1, 1), 1), 202512)

    def test_december_current_month(self):
        self.assertEqual(get_lookback_period(_date(2026, 12, 1), 0), 202612)


class TestConfigValidation(unittest.TestCase):
    def test_rejects_non_first_of_month(self):
        with self.assertRaises(ValueError):
            PreviousPerformanceConfig(rundate=_date(2026, 7, 15))

    def test_rejects_zero_lookback(self):
        with self.assertRaises(ValueError):
            PreviousPerformanceConfig(rundate=_date(2026, 7, 1), lookback_months=0)

    def test_default_lookback_is_twelve(self):
        cfg = PreviousPerformanceConfig(rundate=_date(2026, 7, 1))
        self.assertEqual(cfg.lookback_months, 12)


class TestHistoryStringTruncationBug(unittest.TestCase):
    """Locks in the faithfully-reproduced SAS cat()/length bug (risk #1)."""

    def test_history_string_is_12_chars(self):
        self.assertEqual(len(build_history_string([1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0])), 12)

    def test_history_string_captures_only_first_flag_value_1(self):
        result = build_history_string([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
        self.assertEqual(result, "           1")  # 11 spaces + "1"

    def test_history_string_captures_only_first_flag_value_0(self):
        result = build_history_string([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
        self.assertEqual(result, "           0")  # months 2-12 are NOT reflected — matches the bug

    def test_empty_flags_returns_blank(self):
        self.assertEqual(build_history_string([]), " " * 12)


class TestSummaryFlags(unittest.TestCase):
    def test_no_mia_in_12m(self):
        flags = compute_previous_performance_flags([0] * 12, [0] * 12)
        self.assertEqual(flags["MIA1X_pre12m"], 0)

    def test_mia_at_least_once(self):
        flags = compute_previous_performance_flags([0, 1] + [0] * 10, [0] * 12)
        self.assertEqual(flags["MIA1X_pre12m"], 1)

    def test_mia_multiple_times_still_flags_1x(self):
        # MIA1X_pre12m is a >=1 threshold, not exactly-once, unlike AB1X
        flags = compute_previous_performance_flags([1, 1, 1] + [0] * 9, [0] * 12)
        self.assertEqual(flags["MIA1X_pre12m"], 1)

    def test_arrears_exactly_once(self):
        flags = compute_previous_performance_flags([0] * 12, [0, 1] + [0] * 10)
        self.assertEqual(flags["AB1X_pre12m"], 1)
        self.assertEqual(flags["AB2X_pre12m"], 0)

    def test_arrears_more_than_once(self):
        flags = compute_previous_performance_flags([0] * 12, [1, 1] + [0] * 10)
        self.assertEqual(flags["AB1X_pre12m"], 0)
        self.assertEqual(flags["AB2X_pre12m"], 1)

    def test_arrears_zero_times(self):
        flags = compute_previous_performance_flags([0] * 12, [0] * 12)
        self.assertEqual(flags["AB1X_pre12m"], 0)
        self.assertEqual(flags["AB2X_pre12m"], 0)

    def test_ab1x_and_ab2x_mutually_exclusive_across_range(self):
        for arr_count in range(0, 13):
            arr_flags = [1] * arr_count + [0] * (12 - arr_count)
            flags = compute_previous_performance_flags([0] * 12, arr_flags)
            self.assertFalse(flags["AB1X_pre12m"] == 1 and flags["AB2X_pre12m"] == 1)


class TestQueryBuilder(unittest.TestCase):
    def setUp(self):
        self.cfg = PreviousPerformanceConfig(rundate=_date(2026, 7, 1))

    def test_placeholder_count_matches_param_count(self):
        query = build_monthly_indicator_query(self.cfg)
        params = build_monthly_indicator_params(self.cfg, 202606)
        self.assertEqual(query.count("?"), len(params))

    def test_params_include_excluded_pool_and_period(self):
        params = build_monthly_indicator_params(self.cfg, 202606)
        self.assertIn(self.cfg.excluded_account_pool, params)
        self.assertIn(202606, params)


suite = unittest.TestLoader().loadTestsFromModule(sys.modules[__name__])
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Unit tests failed — see output above."


## 4. Source-to-target reconciliation

In [ ]:
RECONCILIATION_QUERIES = {
    "control_totals": """
        SELECT 'legacy_sas' AS source, COUNT(*) AS row_count,
               SUM(MIA1X_pre12m) AS sum_mia1x, SUM(AB1X_pre12m) AS sum_ab1x, SUM(AB2X_pre12m) AS sum_ab2x
        FROM staging.Prep_{yyyymm}_legacy
        UNION ALL
        SELECT 'python' AS source, COUNT(*) AS row_count,
               SUM(MIA1X_pre12m) AS sum_mia1x, SUM(AB1X_pre12m) AS sum_ab1x, SUM(AB2X_pre12m) AS sum_ab2x
        FROM staging.Prep_{yyyymm}_python;
    """,
    "accounts_in_legacy_not_python": """
        SELECT l.AccountNumber
        FROM staging.Prep_{yyyymm}_legacy l
        LEFT JOIN staging.Prep_{yyyymm}_python p ON l.AccountNumber = p.AccountNumber
        WHERE p.AccountNumber IS NULL;
    """,
    "flag_level_mismatches": """
        SELECT l.AccountNumber,
               l.MIA1X_pre12m AS legacy_MIA1X, p.MIA1X_pre12m AS python_MIA1X,
               l.AB1X_pre12m  AS legacy_AB1X,  p.AB1X_pre12m  AS python_AB1X,
               l.AB2X_pre12m  AS legacy_AB2X,  p.AB2X_pre12m  AS python_AB2X
        FROM staging.Prep_{yyyymm}_legacy l
        INNER JOIN staging.Prep_{yyyymm}_python p ON l.AccountNumber = p.AccountNumber
        WHERE l.MIA1X_pre12m <> p.MIA1X_pre12m
           OR l.AB1X_pre12m  <> p.AB1X_pre12m
           OR l.AB2X_pre12m  <> p.AB2X_pre12m;
        -- NOTE: MIAH/ArrH are deliberately excluded from this comparison; they
        -- are reproduced bug-for-bug (see risk #1) and any legacy value beyond
        -- the first month's flag was never meaningful to begin with.
    """,
}


def render_reconciliation_queries(yyyymm: int) -> dict:
    return {name: sql.format(yyyymm=yyyymm) for name, sql in RECONCILIATION_QUERIES.items()}


## 5. Consolidated `TODO: VERIFY` list

- **`MIAH`/`ArrH` real-world usage** — confirm nothing downstream (report,
  model feature, manual QA check) treats these as genuine 12-month
  history strings; if something does, that's a separate, pre-existing
  data-quality issue to raise with the business, not something to
  silently "fix" during this conversion.
- **Sort-order dependency in the legacy merge** — confirm SQL Server /
  ODBC driver version in production has never changed in a way that
  could have violated the implicit sort assumption in the SAS merge
  loop (see risk #2); the Python version does not share this fragility
  going forward.
- **`ODBC Driver 17 for SQL Server`** — as with Step 1, confirm the
  exact driver available in the target environment.
